In [1]:
print("hello world")

hello world


In [89]:
import pandas as pd
import numpy as np

In [104]:
df = pd.read_csv("earthquake_data.csv")

In [105]:
# Drop rows where target is missing
df = df.dropna(subset=["magnitude"])

# Fill missing values
df["felt"] = df["felt"].fillna(0)
df["cdi"] = df["cdi"].fillna(df["cdi"].median())

In [106]:
df["time"] = pd.to_datetime(df["time"])

df["hour"] = df["time"].dt.hour
df["day"] = df["time"].dt.day
df["month"] = df["time"].dt.month
df["weekday"] = df["time"].dt.weekday

In [107]:
# Distance feature
df["distance_from_center"] = np.sqrt(df["latitude"]**2 + df["longitude"]**2)

# Interaction (SAFE)
df["lat_lon_interaction"] = df["latitude"] * df["longitude"]

In [108]:
drop_cols = [
    "rolling_mag_mean",
    "rolling_mag_std",
    "daily_avg_mag",
    "depth_mag_interaction"   # depends on magnitude → remove
]

df = df.drop(columns=drop_cols, errors="ignore")

In [109]:
# One-hot encoding
df = pd.get_dummies(df, columns=["alert", "status", "magType", "type"], drop_first=True)

In [110]:
df = df.drop(columns=[
    "place",
    "location",
    "country",
    "time",
    "updated"
], errors="ignore")

In [111]:
leakage_cols = ["sig", "cdi", "mmi"]

X = X.drop(columns=leakage_cols, errors="ignore")

In [112]:
X = df.drop(columns=["magnitude"])
y = df["magnitude"]

In [113]:
X = X.select_dtypes(include=["number"])
X = X.fillna(0)

In [114]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [115]:
basic_features = ["depth", "latitude", "longitude", "felt", "cdi"]

X_train_basic = X_train[basic_features]
X_test_basic = X_test[basic_features]

In [116]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

model_before = RandomForestRegressor(random_state=42)
model_before.fit(X_train_basic, y_train)

y_pred_before = model_before.predict(X_test_basic)

print("BEFORE:")
print("R2:", r2_score(y_test, y_pred_before))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_before)))

BEFORE:
R2: 0.8451624529940679
RMSE: 0.46993368424254767


In [117]:
model_after = RandomForestRegressor(random_state=42)
model_after.fit(X_train, y_train)

y_pred_after = model_after.predict(X_test)

print("\nAFTER:")
print("R2:", r2_score(y_test, y_pred_after))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_after)))


AFTER:
R2: 0.9974079808200518
RMSE: 0.060801954493393144


In [118]:
train_pred = model_after.predict(X_train)

print("\nOVERFITTING CHECK:")
print("Train R2:", r2_score(y_train, train_pred))
print("Test R2:", r2_score(y_test, y_pred_after))


OVERFITTING CHECK:
Train R2: 0.9995292442941632
Test R2: 0.9974079808200518


In [119]:
leakage_cols = ["sig", "cdi", "mmi"]

X = X.drop(columns=leakage_cols, errors="ignore")

In [120]:
print(X.columns)

Index(['tz', 'felt', 'tsunami', 'nst', 'dmin', 'rms', 'gap', 'longitude',
       'latitude', 'depth', 'hour', 'day', 'month', 'weekday',
       'distance_from_center', 'lat_lon_interaction'],
      dtype='str')


In [121]:
print(X.corrwith(y).sort_values(ascending=False).head(10))

rms                     0.675313
longitude               0.512984
dmin                    0.494209
nst                     0.490982
lat_lon_interaction     0.384146
depth                   0.370536
gap                     0.114855
tsunami                 0.111727
distance_from_center    0.052531
felt                    0.049741
dtype: float64


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
